In [2]:
library("lme4")
library("margins")
library("stargazer")
library("emmeans")
library("ggeffects")
library("broom")
library("broom.mixed")
library("MASS")
library("pscl")
library("fixest")
library("marginaleffects")
library("modelsummary")
library("glmmTMB")
library("dplyr")

In [3]:
main_path <- "/home/20250114zmz_kd/"
data <- read.csv(paste0(main_path, "UseBSForNot/MergedData1211-2.csv"))
dim(data)

[1] 3227892      79

In [4]:
print(names(data))

 [1] "X"                                           
 [2] "work_id"                                     
 [3] "fac_pub"                                     
 [4] "num_fac"                                     
 [5] "year"                                        
 [6] "paper_type"                                  
 [7] "paper_language"                              
 [8] "nwbin"                                       
 [9] "new_word_reuse_times"                        
[10] "npbin"                                       
[11] "new_phrase_reuse_times"                      
[12] "novelbin"                                    
[13] "rao_stirling"                                
[14] "author_id"                                   
[15] "author_position"                             
[16] "institution_id"                              
[17] "country_code"                                
[18] "country"                                     
[19] "num_author_log"                              
[20] "num_in

In [5]:
colSums(is.na(data))

X 
                                           0 
                                     work_id 
                                           0 
                                     fac_pub 
                                           0 
                                     num_fac 
                                           0 
                                        year 
                                           0 
                                  paper_type 
                                           0 
                              paper_language 
                                           0 
                                       nwbin 
                                           0 
                        new_word_reuse_times 
                                           0 
                                       npbin 
                                           0 
                      new_phrase_reuse_times 
                                           0 
                                    novelbin 
                                           0 
                                rao_stirling 
                                      303430 
                                   author_id 
                                           0 
                             author_position 
                                           0 
                              institution_id 
                                           0 
                                country_code 
                                           0 
                                     country 
                                           0 
                              num_author_log 
                                           0 
                                num_inst_log 
                                           0 
                             num_country_log 
                                           0 
                           num_reference_log 
                                           0 
                             timescited5_log 
                                      107147 
                            timescited10_log 
                                      163886 
                           timescitedall_log 
                                           0 
                               ab_length_log 
                                           0 
                     leadership_global_class 
                                           0 
                         mean_career_age_log 
                                           0 
                            inst_h_index_log 
                                           6 
                                   source_id 
                                           0 
                          source_h_index_log 
                                           0 
                                 source_type 
                                           0 
                                 core_source 
                                           0 
        Agricultural.and.Biological.Sciences 
                                           0 
                         Arts.and.Humanities 
                                           0 
Biochemistry..Genetics.and.Molecular.Biology 
                                           0 
         Business..Management.and.Accounting 
                                           0 
                        Chemical.Engineering 
                                           0 
                                   Chemistry 
                                           0 
                            Computer.Science 
                                           0 
                           Decision.Sciences 
                                           0 
                                   Dentistry 
                                           0 
                Earth.and.Planetary.Sciences 
                                           0 
         Economics..Econometrics.and.Finance 
                                         

In [6]:
# 把所有无限值替换成 NA
data[sapply(data, is.infinite)] <- NA

In [7]:
data <- data[!is.na(data$Atyp_10pct_Z), ]
dim(data)
data <- data[data$num_reference >=5, ]
dim(data)

[1] 2814103      79

[1] 2760385      79

In [8]:
# data <- data[data$ab_length > 0, ]
# dim(data)
data <- data[data$year >= 1950, ]
dim(data)

[1] 2760385      79

In [9]:
# 找出所有包含无限值的行和列
inf_mask <- sapply(data, function(col) is.infinite(col))
rows_with_inf <- apply(inf_mask, 1, any)  # 哪些行至少有一个Inf
cols_with_inf <- colnames(data)[apply(inf_mask, 2, any)]  # 哪些列有Inf

# 打印包含无限值的行数和列名
cat("包含无限值的行数:", sum(rows_with_inf), "\n")
cat("包含无限值的列名:", paste(cols_with_inf, collapse = ", "), "\n")

# 查看这些行具体内容
data_filt_with_inf <- data[rows_with_inf, c(cols_with_inf), drop=FALSE]
print(data_filt_with_inf)

包含无限值的行数: 0 
包含无限值的列名:  
data frame with 0 columns and 0 rows


In [10]:
data$leadership_global_class <- factor(data$leadership_global_class)
data <- within(data, leadership_global_class <- relevel(leadership_global_class, ref = 'shared'))
data$fac_pub <- factor(data$fac_pub)
data <- within(data, fac_pub <- relevel(fac_pub, ref = 'NonBSF'))
data$core_source <- factor(data$core_source)
data <- within(data, core_source <- relevel(core_source, ref = 'NonCore'))
data$year <- as.factor(data$year)

In [11]:
# variable type
paper_level <- "num_author_log + num_inst_log + num_country_log + num_reference_log + leadership_global_class + mean_career_age_log + inst_h_index_log" 
journal_level <- "source_h_index_log + core_source"
disciplines <- "Arts.and.Humanities + Biochemistry..Genetics.and.Molecular.Biology + Business..Management.and.Accounting + Chemical.Engineering + 
 Chemistry + Computer.Science + Decision.Sciences + Dentistry + Earth.and.Planetary.Sciences + 
Economics..Econometrics.and.Finance + Energy + Engineering + Environmental.Science + Health.Professions + 
Immunology.and.Microbiology + Materials.Science + Mathematics + Medicine + Neuroscience + Nursing +
Pharmacology..Toxicology.and.Pharmaceutics + Physics.and.Astronomy + Psychology + Social.Sciences + Veterinary "

In [12]:
fml <- as.formula(
  paste0("novelbin ~ num_fac + ", paper_level, "+", journal_level, "+", disciplines, " | author_id + year")
)

model_total <- feglm(fml, data = data, family = binomial("logit"), vcov = "iid")
summary(model_total)

NOTES: 4 observations removed because of NA values (RHS: 4).
       17,669/0 fixed-effects (78,042 observations) removed because of only 0 (or only 1) outcomes.



GLM estimation, family = binomial, Dep. Var.: novelbin
Observations: 2,682,339
Fixed-effects: author_id: 61,171,  year: 76
Standard-errors: IID 
                                              Estimate Std. Error   z value
num_fac                                       0.062771   0.004901  12.80694
num_author_log                                0.257375   0.004598  55.97065
num_inst_log                                  0.010158   0.005836   1.74062
num_country_log                              -0.282906   0.008630 -32.77986
num_reference_log                             0.154869   0.002965  52.22512
leadership_global_classallNorth               0.133405   0.008758  15.23173
leadership_global_classallSouth              -0.015551   0.013789  -1.12776
mean_career_age_log                           0.031770   0.004870   6.52383
inst_h_index_log                             -0.024365   0.003712  -6.56348
source_h_index_log                           -0.044493   0.002188 -20.33151
core_sourceCore    

In [13]:
pred_curve <- avg_predictions(model_total,
                              variables = list(num_fac = unique(data$num_fac)))
pred_curve

num_fac,estimate,std.error,statistic,p.value,s.value,conf.low,conf.high
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
0,0.3642910,0.005829909,62.48657,0,Inf,0.3528646,0.3757174
1,0.3756048,0.005876809,63.91305,0,Inf,0.3640865,0.3871231
2,0.3870419,0.006051958,63.95317,0,Inf,0.3751803,0.3989035
3,0.3985922,0.006352414,62.74656,0,Inf,0.3861417,0.4110427
4,0.4102451,0.006766804,60.62613,0,Inf,0.3969824,0.4235078
5,0.4219899,0.007278444,57.97803,0,Inf,0.4077244,0.4362553
6,0.4338152,0.007872703,55.10372,0,Inf,0.4183850,0.4492454
7,0.4457097,0.008531025,52.24574,0,Inf,0.4289892,0.4624302
8,0.4576617,0.009240657,49.52696,0,Inf,0.4395503,0.4757730


In [14]:
fname = paste0(main_path, "UseBSForNot/predict_ns_1211_numfac.csv")
write.csv(pred_curve, fname, row.names = FALSE)

In [15]:
paper_vars <- c("num_author_log", "num_inst_log", "num_country_log",
                "num_reference_log", "leadership_global_class",
                "mean_career_age_log", "inst_h_index_log" )


journal_vars <- c("source_h_index_log", "core_source")

disciplines_vars <- c("Arts.and.Humanities", "Biochemistry..Genetics.and.Molecular.Biology", "Business..Management.and.Accounting",
                 "Chemical.Engineering", "Chemistry", "Computer.Science", "Decision.Sciences", "Dentistry",
                 "Earth.and.Planetary.Sciences", "Economics..Econometrics.and.Finance", "Energy", "Engineering",
                 "Environmental.Science + Health.Professions", "Immunology.and.Microbiology", "Materials.Science", "Mathematics",
                 "Medicine", "Neuroscience", "Nursing", "Pharmacology..Toxicology.and.Pharmaceutics", "Physics.and.Astronomy",
                 "Psychology", "Social.Sciences", "Veterinary")

In [16]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_total,
                           keep = c("num_fac", paper_vars, journal_vars),
                           se = "iid",
                           # cluster = ~ paper_year + paper_field,
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                 & novelbin\\  
   Model:                              & (1)\\  
   \midrule
   \emph{Variables}\\
   num\_fac                            & 0.063$^{***}$\\   
                                       & (0.005)\\   
   num\_author\_log                    & 0.257$^{***}$\\   
                                       & (0.005)\\   
   num\_inst\_log                      & 0.010$^{*}$\\   
                                       & (0.006)\\   
   num\_country\_log                   & -0.283$^{***}$\\   
                                       & (0.009)\\   
   num\_reference\_log                 & 0.155$^{***}$\\   
                                       & (0.003)\\   
   leadership\_global\_classallNorth   & 0.133$^{***}$\\   
                                       & (0.009)\\   
   leadership\_global\_classallSouth   & -0.016\\   
                                       & (0.014

In [17]:
fml <- as.formula(
  paste0("novelbin ~ fac_pub + ", paper_level, "+", journal_level, "+", disciplines, " | year")
)

model_total <- feglm(fml, data = data, family = binomial("logit"), vcov = "iid")
summary(model_total)

NOTE: 4 observations removed because of NA values (RHS: 4).



GLM estimation, family = binomial, Dep. Var.: novelbin
Observations: 2,760,381
Fixed-effects: year: 76
Standard-errors: IID 
                                              Estimate Std. Error   z value
fac_pubBSF                                    0.107377   0.004467  24.03723
num_author_log                                0.121413   0.003351  36.23585
num_inst_log                                 -0.024123   0.004387  -5.49837
num_country_log                              -0.207627   0.006648 -31.23053
num_reference_log                             0.094582   0.002382  39.69979
leadership_global_classallNorth               0.084862   0.007054  12.03008
leadership_global_classallSouth              -0.212649   0.007916 -26.86415
mean_career_age_log                          -0.032832   0.003231 -10.16102
inst_h_index_log                             -0.007020   0.002262  -3.10397
source_h_index_log                           -0.119054   0.001773 -67.13381
core_sourceCore                        

In [18]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_total,
                           keep = c("fac_pub", paper_vars, journal_vars),
                           se = "iid",
                           # cluster = ~ paper_year + paper_field,
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                 & novelbin\\  
   Model:                              & (1)\\  
   \midrule
   \emph{Variables}\\
   fac\_pubBSF                         & 0.107$^{***}$\\   
                                       & (0.004)\\   
   num\_author\_log                    & 0.121$^{***}$\\   
                                       & (0.003)\\   
   num\_inst\_log                      & -0.024$^{***}$\\   
                                       & (0.004)\\   
   num\_country\_log                   & -0.208$^{***}$\\   
                                       & (0.007)\\   
   num\_reference\_log                 & 0.095$^{***}$\\   
                                       & (0.002)\\   
   leadership\_global\_classallNorth   & 0.085$^{***}$\\   
                                       & (0.007)\\   
   leadership\_global\_classallSouth   & -0.213$^{***}$\\   
                                    